# 事前準備：共通コードの実行
* このノートブックに接続したら，まずは以下の2つの共通コード（コードAとコードB）を実行する
* これらの共通コードを実行しないと，それ以降のコードが実行できないので注意する
* また，コードA，コードB，及びコードC は，ノートブックに接続するたび毎回実行すること（ノートブックに接続中は，何度も実行する必要はない）
* 共通コードの詳細についての説明は割愛する（簡単な説明は第2回の「[サンプルノートブック02](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook02.ipynb)」を参照）

In [ ]:
# コードA：日本語化ライブラリ導入
! pip install japanize-matplotlib | tail -n 1

In [ ]:
# コードB：共通事前処理

# B1:余分なワーニングを非表示にする
import warnings
warnings.filterwarnings('ignore')

# 必要ライブラリのimport
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib # matplotlib日本語化対応
import seaborn as sns

# B2:データフレーム表示用関数
from IPython.display import display

# B3:表示オプション調整
np.set_printoptions(suppress = True, precision = 3) #numpyの浮動小数点の表示精度
pd.options.display.float_format = '{:.3f}'.format #pandasでの浮動小数点の表示精度
pd.set_option('display.max_columns', None) #データフレームですべての列データを表示

# B4:グラフのデフォルトフォント指定
plt.rcParams['font.size'] = 14

# 乱数の種
random_seed = 123

In [ ]:
# コードC：データのダウンロード
!curl -L -o Survived.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/04/Survived.csv
!curl -L -o employee_train.csv https://raw.githubusercontent.com/yoshida-nu/lecture_public/main/datascience/resources/04/employee_train.csv

# 目的と到達目標

## 目的
* 過学習について理解する
* ダミー変数化を用いた分類木モデルの学習について理解する

## 到達目標
* 過学習について分類木モデルの学習を例にして説明できる
* ダミー変数化を使った分類木による一連の分析を実践できる


# 分類木によるデータ分析例 その1

## 用いるデータと分析の目的
* 例として，csvファイル「Survived.csv」を用いる
* このデータは，過去に沈没した客船の乗客に関する情報がまとめられたデータ（客船データと呼ぶ）である
  * 出典: 須藤秋良, 株式会社フレアリンク: スッキリわかるPythonによる機械学習入門, インプレス, 2020

|**列名**| **意味** |
|:--|:--|
|乗客ID| 乗客のID |
|生存| 0: 死亡 / 1: 生存 |
|チケットクラス| チケットの等級（1: 1等級 / 2: 2等級 / 3: 3等級） |
|性別| 乗客の性別（female: 女性 / male: 男性） |
|年齢| 乗客の年齢 |
|同乗1| 同乗した兄弟と配偶者の数 |
|同乗2| 同乗した親と子供の数 |
|チケットID| チケットのID |
|運賃| 乗船運賃 |
|部屋番号| 乗客の部屋番号 |
|港コード| 乗船した港を表すアルファベット |

* 客船データを用いて，「生存」を予測する分類木モデルを作成することを分析の目的とする
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `display`関数を使ってデータ（`df`の内容）を表示


In [ ]:
# データの読み込み
df = pd.read_csv('Survived.csv')
display(df)

## 各カテゴリのデータ数の確認
* 今回は，生存列を目的変数とした分類を考える ⇒ カテゴリは「0: 死亡」と「1: 生存」の2つ
* まず，生存列の各カテゴリのデータ数を確認する
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: 列「生存」の値の頻度を抽出し，その結果を`display`関数で表示
  * `df['生存'].value_counts()`で，DataFrameである`df`の列「生存」にどんな値がそれぞれ何個あるのかを調べる
  * `value_counts`メソッドの戻り値がSeriesなので，pandasの`DataFrame`関数でDataFrameに変換 ⇒ `pd.DataFrame(df['種類'].value_counts())`
  * 変換したDataFrameを`display`関数で表示

In [ ]:
# 各カテゴリのデータ数の確認
df = pd.read_csv('Survived.csv')
display(pd.DataFrame(df['生存'].value_counts()))

## 不均衡データ
* 目的変数とする生存列のデータは「0（死亡）」が549個で，「1（生存）」が342個と偏りがある ⇒ 不均衡データと呼ぶ
* 不均衡データだとうまく学習できないことがある
  * 極端な例として，生存者が全体の5%だとする
  * 年齢が0以上という条件で分類しても95%は正解になる
  * この場合，実質的に学習しなくてもうまく分類できていることになる
* 学習時に不均衡データに対する対応が必要（後述）

  <img src="./fig/imbalanced_data.jpg" width="250">

## 欠損値の確認
* DataFrameに対して`isnull`メソッドを適用することで，欠損値の場所が`True`となる
* さらに，`isnull`メソッドの戻り値となるDataFrameに対して，`sum`メソッドを適用することで，各列の`True`（欠損値）の数が確認できる
* `sum`メソッドは合計を計算するメソッドであるが，計算対象のデータ型がboolの場合は，`True`を1，`False`を0として計算する
* 書式: `df.isnull().sum()`
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `isnull`メソッドと`sum`メソッドを組み合わせて，各列の欠損値の数を計算して，`display`関数で表示する

In [ ]:
# 欠損値の確認
df = pd.read_csv('Survived.csv')
display(pd.DataFrame(df.isnull().sum(), columns=['欠損値の数']))

## 欠損値への対処
* 部屋番号列の欠損値が非常に多いので，説明変数として使用しないことにする
* また，港コードも説明変数から除外する
* 一方，年齢列のデータも欠損値が多く存在するが，生存列の値に関係がありそうなので，説明変数として使用することにする
* そこで，年齢列の欠損値を算術平均（`df['年齢'].mean()`）に置き換える
* 欠損値の置き換えは，`fillna`メソッドを用いる
* 書式: `fillna(置き換える値)`
  
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `fillna`メソッドを使って，年齢列（`df['年齢']`）の欠損値を，その算術平均`df['年齢'].mean()`に置き換える
* 4行目: `isnull`メソッドと`sum`メソッドを組み合わせて，各列の欠損値の数を計算して，`display`関数で表示する

In [ ]:
# 欠損値の補完
df = pd.read_csv('Survived.csv')
df['年齢'] = df['年齢'].fillna(df['年齢'].mean())
display(pd.DataFrame(df.isnull().sum(), columns=['欠損値の数']))

## 実習：データの分割
* これまでと同様にして，DataFrame `df` を説明変数`x`と目的変数`t`に分割する  
* 説明変数には「チケットクラス」「年齢」「同乗1」「同乗2」「運賃」を用いる
* さらに，`x`と`t`を訓練データ`(x_train, t_train)`とテストデータ`(x_test, t_test)`に分割する
* 詳細は，[第3回のサンプルノートブック](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook03.ipynb)を参照

**［実習内容］**
* 以下の「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: `fillna`メソッドを使って，年齢列（`df['年齢']`）の欠損値を，その算術平均`df['年齢'].mean()`に置き換える
* 4行目: 説明変数の列名を要素とするリスト`['チケットクラス', '年齢', '同乗1', '同乗2', '運賃']`を変数`x_cols`に代入
* 5行目: 目的変数の列名（リスト）`['生存']`を変数`t_col`に代入
* 6行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 7行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 8行目: `model_selection`モジュールの`train_test_split`関数のインポート
* 9行目: `train_test_split`関数を使って説明変数`x`と目的変数`t`を訓練データ（80％）とテストデータ（20％）にそれぞれ分割
  * `test_size=0.2`として，訓練データを80％，テストデータを20％に分割
  * `random_state=random_seed`として，乱数を固定する ⇒ 結果が同じになる
* 10行目: `print`関数で区切り線「================ x_train ================」を表示
* 11行目: `display`関数で変数`x_train`（訓練データの説明変数）の先頭2行を表示
* 12行目: `print`関数で区切り線「================ t_train ================」を表示
* 13行目: `display`関数で変数`t_train`（訓練データの目的変数）の先頭2行を表示
* 14行目: `print`関数で区切り線「================ x_test ================」を表示
* 15行目: `display`関数で変数`x_test`（テストデータの説明変数）の先頭2行を表示
* 16行目: `print`関数で区切り線「================ t_test ================」を表示
* 17行目: `display`関数で変数`t_test`（テストデータの目的変数）の先頭2行を表示

**［実行結果］**

  <img src="./fig/exercise_Survived_Data_splitting.jpg" width="350">

In [ ]:
# データの分割
df = pd.read_csv('Survived.csv')
df['年齢'] = df['年齢'].fillna(df['年齢'].mean())






print('================ x_train ================')
display(x_train.head(2))
print('================ t_train ================')
display(t_train.head(2))
print('================ x_test ================')
display(x_test.head(2))
print('================ t_test ================')
display(t_test.head(2))

## 分類木モデルの学習と評価（1）

### 分類木モデルの学習と不均衡データへの対応
* 目的変数とする生存列のデータは「0（死亡）」が549個で，「1（生存）」が342個と偏りがある
* このような偏りのあるデータのことを不均衡データと呼ぶ
* 一般的に，不均衡データを使ってモデルの学習を行うと，良いモデルにならないことがある
* 極端な例として， 生存者が全体の5%だとすると，年齢が0以上という条件で分類しても95%は正解になる
* この場合，実質的に学習しなくてもうまく分類できていることになるが，これでは生存に影響を与えている要因（説明変数）を明らかにすることができないため，（精度は高いが）良いモデルとは言えない
* 不均衡データを使って学習する場合には，データの偏りの影響を取り除くための対応が必要となる
* 具体的には，`sklearn`（scikit-learn）の`tree`モジュールにおける`DecisionTreeClassifier`クラスから分類木のモデルオブジェクトを生成する際に，不均衡データの対処をするための引数`class_weight='balanced'`を指定する
* 詳細は省略するが，これによって，比率の大きいデータの影響を小さくし，比率の小さいデータの影響を大きくすることができる
* そのため，不均衡データの影響を取り除くことができる

### 実習：分類木モデルの学習と評価
* 第3回と同様にして，分類木モデルの学習と評価を行う
* 詳細は，[第3回のサンプルノートブック](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook03.ipynb)を参照
* また，今回は訓練データとテストデータの両方で精度を計算し，比較する

**［実習内容］**
* 以下の「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2～9行目: ファイルの読み込み，欠損値の対処，データの分割（上のコードと同じ）
* 10行目: `sklearn` (scikit-learn) の`tree`モジュールの`DecisionTreeClassifier`クラスをインポート
* 11行目: 分類木モデルの学習を行うためのオブジェクトを`DecisionTreeClassifier`クラスから生成し，変数`model_tree`に代入
  * `class_weight='balanced'`で不均衡データの影響を取り除く
  * `max_depth=5`で最大深さを5とする
  * `random_state=random_seed`で乱数の種を指定する
* 12行目: `fit`メソッドで分類木モデルの学習を実行
  * 学習には訓練データ`x_train`, `t_train`を使う
* 13行目: `model_tree.score(X=x_train, y=t_train)`で，訓練データから精度を計算して変数`score_train`に代入
* 14行目: `model_tree.score(X=x_test, y=t_test)`で，テストデータに対する精度を計算して変数`score_test`に代入
* 15行目: `print`関数とf-stringを使って，訓練データに対する精度（`score_train`）とテストデータに対する精度（`score_test`）を表示
  * 「`:.3f`」で，小数点以下3桁まで表示

**［実行結果］**
```
訓練データの精度=0.716 / テストデータの精度=0.665
```

In [ ]:
# モデルの学習
df = pd.read_csv('Survived.csv')
df['年齢'] = df['年齢'].fillna(df['年齢'].mean())







model_tree = DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=random_seed)


score_test = model_tree.score(X=x_test, y=t_test)
print(f'訓練データの精度={score_train:.3f} / テストデータの精度={score_test:.3f}')

### 最大深さによる精度の変化の確認と過学習
* これまでは，分類木モデルの学習の際には最大深さを適当に固定していた
* 分類木モデルの性能は最大深さが大きく影響するので，十分に検討した上で決める必要がある
* 以下は，分類木モデルの最大深さを1から14まで変化させたときの，訓練データとテストデータのそれぞれで計算した精度の推移の表とグラフである
  
|**深さ**| **訓練データの精度** | **テストデータの精度** |
|:--|:--|:--|
|1|0.618|0.598|
|2|0.653|0.642|
|3|0.654|0.648|
|4|0.636|0.581|
|5|0.716|0.665|
|6|0.721|0.665|
|7|0.767|0.682|
|8|0.774|0.659|
|9|0.810|0.704|
|10|0.824|0.642|
|11|0.843|0.642|
|12|0.879|0.676|
|13|0.900|0.659|
|14|0.916|0.648|

<img src="./fig/accuracy_graph.jpg" width="500">  

* 分類木の最大深さを大きくすると，分岐条件を多く設定することができるので，学習データを細かく分類することができる
* 分類木モデルは最大深さを大きくとることで複雑なモデルが表現できるので，表現力が豊かなモデルになる
* このことは訓練データの正解率の推移から確認できる（最大深さを大きくすると正解率も大きくなっている）
* 一方，テストデータの正解率は，最大深さが深くなってくると減少傾向になる
* この現象は，モデルを複雑にする（最大深さを大きくする）ことで，訓練データの詳細な特徴まで学習してしまい，訓練データに対して過度に当てはまったモデルになってしまったことが原因で生じている
* この現象のことを一般に**過学習**と呼ぶ
* 過学習によって，未知のデータ（テストデータ）に対して当てはまりが悪くなるので，テストデータの正解率は小さくなってしまう
* 以上より，モデルを必要以上に複雑にすると未知のデータに対するモデルの性能が低下するので，適切なモデルの複雑さにしてモデルを学習する必要がある
* また，過学習は分類木モデルだけでなく，すべての教師あり学習のモデルで起こる現象である
* 例えば，線形重回帰モデルでは説明変数の種類を増やすとモデルが複雑になり（表現力が豊かになり）過学習を起こしやすくなる


## 分類木モデルの学習と評価（2）

### モデルの性能向上
* 一般に，過学習を回避しつつ，良いモデル（未知データに対して当てはまりがよいモデル）を作るためには，様々なアプローチがある
* ここでは，まず欠損値の対処方法の改善，及び使用する説明変数の再検討を試みる




### 年齢の分布に関するクロス集計
* 先ほどは年齢列の欠損値を算術平均に置き換えて対処していたが，この対処を改善することにする
* ここでは，年齢の分布がチケットクラスや生存者/死亡者（生存）の違いによって異なると考え，これら説明変数の値でグループに分け，それぞれのグループでの平均年齢（年齢の算術平均）を確認する
* この分析を一般には「クロス集計」と呼ぶ
* DataFrameに対するクロス集計は，`pivot_table`関数を用いる
* 詳細は教科書及び共通資料「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
* ピボットテーブル集計の書式：`pd.pivot_table(df, index=集計軸の列名1, columns=集計軸の列名2, values=集計対象の列, aggfunc=関数（のリスト）)`
  * `aggfunc`のデフォルトは`mean`
  * `aggfunc`は，aggregate function（集約関数）の略
  
**［以下のコードの処理内容］**
* 2行目: ファイルの読み込み
* 3行目: `pivot_table`関数を使って，チケットクラスと生存の値の組み合わせによる年齢の算術平均を集計し，`display`関数で表示
  * 引数は `index='生存'`, `columns='チケットクラス'`, `values='年齢'` とする
  * `aggfunc`はデフォルト（算術平均`mean`）を使用するので指定しない


In [ ]:
# クロス集計
df = pd.read_csv('Survived.csv')
display(pd.pivot_table(df, index='生存', columns='チケットクラス', values='年齢'))

In [ ]:
score_df=pd.DataFrame([[90, 70], [70, 80], [70, 80], [85, 70]],
                        index=['工藤', '浅木', '松田', '福田'],
                        columns=['Python', 'ML'])

print(score_df['ML'] == 70)

### 欠損値への対処の改善
* 上のコードの結果から，チケットクラス（1～3）と生存（0 or 1）の組合せで平均年齢が違っている
* そこで，年齢列の欠損値に対しては，該当する算術平均（小数点以下は切り捨てる）でそれぞれ置き換えることにする
* この置き換えには，DataFrameの`loc`メソッドを用いる
* 詳細は教科書及び共通資料「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
* 具体的には，`df.loc[[条件式], '年齢'] = 置き換える値`とし，条件式を満たすデータの年齢の値を置き換える
* チケットクラス（1～3）と生存（0 or 1）の組み合わせの数の`[条件式]`を指定する
* 組合せは以下の6通りとなる
  * チケットクラスが「1（1等級）」で生存が「0（死亡）」
  * チケットクラスが「1（1等級）」で生存が「1（生存）」
  * チケットクラスが「2（2等級）」で生存が「0（死亡）」
  * チケットクラスが「2（2等級）」で生存が「1（生存）」
  * チケットクラスが「3（3等級）」で生存が「0（死亡）」
  * チケットクラスが「3（3等級）」で生存が「1（生存）」
* 以下のコードでは，`loc`メソッドの条件式を`(df['チケットクラス'] == x) & (df['生存'] == y) & (df['年齢'].isnull())`としている
  * ここで，`x`には 1, 2, 3 のいずれかの値，`y`には 0, 1 のいずれかの値が入る
  * 条件式の意味は「チケットクラスの値が`x`，かつ生存の値が`y`，かつ年齢が欠損値となっている」となる
  * 条件式が成立する（`True`の）データに対して置き換えが行われる
  * 条件式内の`df['チケットクラス'] == x`，`df['生存'] == y`，及び`df['年齢'].isnull()`は，値が`True`と`False`からなる論理配列となる
  * したがって，条件式が`True`となるデータ（行データ）は，`df['チケットクラス'] == x`，`df['生存'] == y`，及び`df['年齢'].isnull()`がすべて`True`となる位置のデータに対応する
  * 論理配列の作成については，教科書及び共通資料「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
  
**［以下のコードの処理内容］**
* 2行目: ファイルの読み込み
* 3行目: チケットクラスが1，かつ生存が0，かつ年齢が欠損している（`NaN`の）データの年齢列の値を43にする
  * 条件式は`(df['チケットクラス'] == 1) & (df['生存'] == 0) & (df['年齢'].isnull())`
* 4行目: チケットクラスが1，かつ生存が1，かつ年齢が欠損している（`NaN`の）データの年齢列の値を35にする
  * 条件式は`(df['チケットクラス'] == 1) & (df['生存'] == 1) & (df['年齢'].isnull())`
* 5行目: チケットクラスが2，かつ生存が0，かつ年齢が欠損している（`NaN`の）データの年齢列の値を33にする
  * 条件式は`(df['チケットクラス'] == 2) & (df['生存'] == 0) & (df['年齢'].isnull())`
* 6行目: チケットクラスが2，かつ生存が1，かつ年齢が欠損している（`NaN`の）データの年齢列の値を25にする
  * 条件式は`(df['チケットクラス'] == 2) & (df['生存'] == 1) & (df['年齢'].isnull())`
* 7行目: チケットクラスが3，かつ生存が0，かつ年齢が欠損している（`NaN`の）データの年齢列の値を26にする
  * 条件式は`(df['チケットクラス'] == 3) & (df['生存'] == 0) & (df['年齢'].isnull())`
* 8行目: チケットクラスが3，かつ生存が1，かつ年齢が欠損している（`NaN`の）データの年齢列の値を20にする
  * 条件式は`(df['チケットクラス'] == 3) & (df['生存'] == 1) & (df['年齢'].isnull())`
* 9行目: `isnull`メソッドと`sum`メソッドを組み合わせて，各列の欠損値の数を計算して，`display`関数で表示する

In [ ]:
# 欠損値の補完（条件付き平均値）
df = pd.read_csv('Survived.csv')
df.loc[(df['チケットクラス'] == 1) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 43
df.loc[(df['チケットクラス'] == 1) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 35
df.loc[(df['チケットクラス'] == 2) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 33
df.loc[(df['チケットクラス'] == 2) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 25
df.loc[(df['チケットクラス'] == 3) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 26
df.loc[(df['チケットクラス'] == 3) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 20
display(pd.DataFrame(df.isnull().sum()))

### 性別列と生存列の関係
* ここで，性別（female/male）と生存者（生存列の値が1）の割合に関係があるかを調べる
* 具体的には，性別ごとの生存者の割合（生存列の算術平均と等価）を計算する
* ある列名の値によって分けられるグループごとに統計量等を計算する場合には，`groupby`メソッドを使う
* 詳細は教科書及び共通資料「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
* 基本的な書式は`df.groupby('グループ分けしたい列インデックス').メソッド名()`となる．  
* 例えば，`df`の中の性別列の値（female/male）ごとに算術平均を計算したい場合には，算術平均が計算不可能な列「チケットID」「部屋番号」「港コード」を削除したうえで，`df.groupby('性別').mean()`と記述する
* ただし，この記述だとすべての列データに対して算術平均を計算することになるので，例えば，生存列の算術平均だけ取り出したい場合は，`df.groupby('性別').mean()['生存']`と記述する
  
**［以下のコードの処理内容］**
* 2行目: ファイルの読み込み
* 3行目: `drop`メソッドを使って，`df`から「チケットID」「部屋番号」「港コード」の列（`axis=1`）を削除
  * 書式: `df.drop([削除する列名のリスト], axis=1)`
* 4行目: `groupby`メソッドを使って，性別ごとの生存者の割合を計算し，`display`メソッドで表示
  * 生存者の割合は生存列の算術平均で計算できるので，`mean`メソッドを使う ⇒ `df.groupby('性別').mean()['生存']`
  * `groupby`メソッドによる集計結果は Series となるので，それを`DataFrame`関数でDataFrameに変換してから表示

In [ ]:
# 性別ごとの生存率
df = pd.read_csv('Survived.csv')
df = df.drop(['チケットID', '部屋番号', '港コード'], axis = 1)
display(pd.DataFrame(df.groupby('性別').mean()['生存']))

### 性別列のダミー変数化（ワンホットエンコーディング）
* scikit-learnを使ったモデルの学習では，数値（boolを含む）のデータ列しか利用できないので，性別列のデータはこのままでは使えない
* そこで，性別列の「female」と「male」を1と0，あるいは`True`と`False`などの数値として扱える値に変換して，学習に利用できるようにする
* この操作のことを「**ダミー変数化**（またはワンホットエンコーディング）」と呼ぶ
* ダミー変数化には，pandasの`get_dummies`関数を用いる
  * 書式: `pd.get_dummies(ダミー変数化する列)`
  * 詳細は教科書及び共通資料「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
* 具体的には，以下の処理を行う．
  * female列とmale列を新たに追加
  * female列: 性別列の値が「female」の場合は`True` または 1，「male」の場合は`False` または 0に変換
  * male列: 性別列の値が「male」の場合は`True` または 1，「female」の場合は`False` または 0に変換
* 上記処理でfemale列とmale列の2列を追加しているが，一方の列が0（`False`）であれば，もう一方の列は必ず1（`True`）になる（逆も同様）
* したがって，モデルの学習に利用するのはどちらかの列で十分であることに注意しておく
  
**［以下のコードの処理内容］**
*  2行目: ファイルの読み込み
*  3行目: `get_dummies`関数を使って，性別列（`df['性別']`）をダミー変数化し，その結果を変数`dummy`に代入
*  4行目: `display`関数を使って，変数`dummy`の内容を表示

In [ ]:
# 性別のダミー変数化
df = pd.read_csv('Survived.csv')
dummy = pd.get_dummies(df['性別'])
display(dummy)

### DataFrameの結合
* 客船データの`df`（DataFrame）と性別をダミー変数化した`dummy`（DataFrame）を結合して1つのDataFrameとして扱えるようにする
* DataFrameの結合には，`concat`関数を用いる
* 詳細は教科書及び共通資料「[ノートブック：SeriesとDataFrameの基本操作](https://colab.research.google.com/github/yoshida-nu/lecture_public/blob/main/datascience/doc/datascience_notebook_se_df.ipynb)」を参照
* `concat`関数の書式： `pd.concat([df1, df2], axis = [0 or 1])`
  * `df1`と`df2`は結合するDataFrameの名前
  * `axis=0` ⇒ 行方向の結合（縦に並べて結合）
  * `axis=1` ⇒ 列方向の結合（横に並べて結合）
  
**［以下のコードの処理内容］**
* 2行目: ファイルの読み込み
* 3行目: `get_dummies`関数を使って，性別列（`df['性別']`）をダミー変数化し，その結果を変数`dummy`に代入
* 4行目: `concat`関数を使って，2つのDataFrame`df`と`dummy`を結合し，変数`df`に代入
  * 列方向の結合（横に並べて結合）なので，`axis=1` とする
* 5行目: `display`関数を使って，変数`df`の内容を表示

In [ ]:
# 性別のダミー変数化（元のDataFrameに結合）
df = pd.read_csv('Survived.csv')
dummy = pd.get_dummies(df['性別'])
df = pd.concat([df, dummy], axis=1)
display(df)

### 説明変数に性別列（female列）を追加
* これまでと同様にして，客船データのDataFrame「`df`」を訓練データ`(x_train, t_train)`とテストデータ`(x_test, t_test)`に分割する
* 先ほどは，説明変数に，チケットクラス列，年齢列，同乗1列，同乗2列，運賃列の5個を選択していたが，ここでは新たにfemale列を追加して合計6個の説明変数とする
* 年齢列の欠損値への対処も同様の処理を行う
  
**［以下のコードの処理内容］**
* 2～4行目: ファイルの読み込み，ダミー変数化とデータの結合
* 5～10行目: 年齢列の欠損値への対処
* 11行目: 説明変数の列名を要素とするリスト`['チケットクラス', '年齢', '同乗1', '同乗2', '運賃', 'female']`を変数`x_cols`に代入
* 12行目: 目的変数の列名（リスト）`['生存']`を変数`t_col`に代入
* 13行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 14行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 15行目: `model_selection`モジュールの`train_test_split`関数の読み込み
* 16行目: `train_test_split`関数を使って説明変数`x`と目的変数`t`を訓練データ（80％）とテストデータ（20％）にそれぞれ分割
* 17～24行目: 各データの先頭3行を表示

In [ ]:
# データの分割（性別のダミー変数化後）
df = pd.read_csv('Survived.csv')
dummy = pd.get_dummies(df['性別'])
df = pd.concat([df, dummy], axis=1)
df.loc[(df['チケットクラス'] == 1) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 43
df.loc[(df['チケットクラス'] == 1) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 35
df.loc[(df['チケットクラス'] == 2) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 33
df.loc[(df['チケットクラス'] == 2) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 25
df.loc[(df['チケットクラス'] == 3) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 26
df.loc[(df['チケットクラス'] == 3) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 20
x_cols = ['チケットクラス', '年齢', '同乗1', '同乗2', '運賃', 'female']
t_col = ['生存']
x = df[x_cols]
t = df[t_col]
from sklearn.model_selection import train_test_split
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size = 0.2, random_state = random_seed)
print('================ x_train ================')
display(x_train.head(3))
print('================ t_train ================')
display(pd.DataFrame(t_train.head(3)))
print('================ x_test ================')
display(x_test.head(3))
print('================ t_test ================')
display(pd.DataFrame(t_test.head(3)))

### 分類木モデルの学習と評価
* 前のコードで作成したデータを使ってモデルの学習と評価を再度行う
  
**［以下のコードの処理内容］**
* 2～16行目: ファイルの読み込み，ダミー変数化とデータの結合，データの分割
* 17行目: `sklearn` (scikit-learn) の`tree`モジュールを読み込む 
* 18行目: `tree.DecisionTreeClassifier`で，分類木モデルの学習を行うためのオブジェクトを`tree`モジュールの`DecisionTreeClassifier`クラスから生成し，変数`model_tree`に代入
>* `class_weight = 'balanced'`で不均衡データの影響を取り除く
>* `max_depth = 5`で最大深さを5とする
>* `random_state = random_seed`で乱数の種を指定する
* 19行目: `fit`メソッドで分類木モデルの学習を実行
>* 学習には訓練データ`x_train`, `t_train`を使う
* 20行目: `model_tree.score(X = x_train, y = t_train)`で，訓練データから精度を計算して変数`score_train`に代入
* 21行目: `model_tree.score(X = x_test, y = t_test)`で，テストデータに対する精度を計算して変数`score_test`に代入
* 22行目: `print`関数とf-stringを使って，訓練データに対する精度（`score_train`）とテストデータに対する精度（`score_test`）を表示
>* 「`:.3f`」で，小数点以下3桁まで表示

In [ ]:
# モデルの学習（性別のダミー変数化後）
df = pd.read_csv('Survived.csv')
dummy = pd.get_dummies(df['性別'])
df = pd.concat([df, dummy], axis=1)
df.loc[(df['チケットクラス'] == 1) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 43
df.loc[(df['チケットクラス'] == 1) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 35
df.loc[(df['チケットクラス'] == 2) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 33
df.loc[(df['チケットクラス'] == 2) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 25
df.loc[(df['チケットクラス'] == 3) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 26
df.loc[(df['チケットクラス'] == 3) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 20
x_cols = ['チケットクラス', '年齢', '同乗1', '同乗2', '運賃', 'female']
t_col = ['生存']
x = df[x_cols]
t = df[t_col]
from sklearn.model_selection import train_test_split
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size = 0.2, random_state = random_seed)
from sklearn.tree import DecisionTreeClassifier
model_tree = DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=random_seed)
model_tree.fit(X=x_train, y=t_train)
score_train = model_tree.score(X=x_train, y=t_train)
score_test = model_tree.score(X=x_test, y=t_test)
print(f'訓練データの精度={score_train:.3f} / テストデータの精度={score_test:.3f}')

### 最大深さによる精度の変化の確認
* 先ほどと同様にして，訓練データとテストデータに対する各精度の推移を確認する
* 最大深さが４のときに，テストデータの精度が最大となった
  
|**深さ**| **訓練データの精度** | **テストデータの精度** |
|:--|:--|:--|
|1|0.784|0.799|
|2|0.792|0.793|
|3|0.834|0.860|
|4|0.851|0.872|
|5|0.862|0.827|
|6|0.888|0.849|
|7|0.912|0.821|
|8|0.930|0.832|
|9|0.952|0.827|
|10|0.963|0.793|
|11|0.975|0.793|
|12|0.978|0.804|
|13|0.978|0.782|
|14|0.983|0.799|

<img src="./fig/accuracy_graph2.jpg" width="500">  


### 目的変数に影響している説明変数の確認
* 目的変数である生存列に影響を与えている説明変数を確認する
* 目的変数への影響の大きさを測る代表的な指標に特徴量重要度がある
* 特徴量重要度は0～1の値をとり，1に近いほどその列が目的変数の予測に与える影響が大きい
* 特徴量重要度は学習済みのオブジェクト`model_tree`における`feature_importances_`属性を参照することで確認できる
  
**［以下のコードの処理内容］**
* 2～18行目: ファイルの読み込み，ダミー変数化とデータの結合，データの分割，モデルの学習
* 19行目: `display`関数で，`model_tree`オブジェクトの`feature_importances_`属性を表示
  * `DataFrame`関数で，`model_tree.feature_importances_` をDataFrameに変換
  * そのDataFrameの行名（`index`）を説明変数`x`の列名（`x.columns`），列名（`columns`）を「特徴量重要度」とした

In [ ]:
# 特徴量重要度の表示
df = pd.read_csv('Survived.csv')
dummy = pd.get_dummies(df['性別'])
df = pd.concat([df, dummy], axis=1)
df.loc[(df['チケットクラス'] == 1) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 43
df.loc[(df['チケットクラス'] == 1) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 35
df.loc[(df['チケットクラス'] == 2) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 33
df.loc[(df['チケットクラス'] == 2) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 25
df.loc[(df['チケットクラス'] == 3) & (df['生存'] == 0) & (df['年齢'].isnull()), '年齢'] = 26
df.loc[(df['チケットクラス'] == 3) & (df['生存'] == 1) & (df['年齢'].isnull()), '年齢'] = 20
x_cols = ['チケットクラス', '年齢', '同乗1', '同乗2', '運賃', 'female']
t_col = ['生存']
x = df[x_cols]
t = df[t_col]
from sklearn.model_selection import train_test_split
x_train, x_test, t_train, t_test = train_test_split(x, t, test_size = 0.2, random_state = random_seed)
from sklearn.tree import DecisionTreeClassifier
model_tree = DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=random_seed)
model_tree.fit(X=x_train, y=t_train)
display(pd.DataFrame(model_tree.feature_importances_, index=x.columns, columns=['特徴量重要度']))

## 実習：一連の分析
* 最後にまとめとして，これまでの分析を一括で行う
  
**［分析手順］**
* ホールドアウト法による分類木モデルの学習を行う
  * 目的変数: 生存 
  * 説明変数（6つ）: チケットクラス, 年齢, 同乗1, 同乗2, 運賃, female
  * 訓練データ 80％，テストデータ 20％として分割
  * 不均衡データの影響を取り除く
  * 木の最大深さは 5 とする
  * 乱数の種（`random_state`）は，すべて共通コードで定義している`random_seed`を用いる
* 学習したモデルを使って予測を行う
  * 予測する客船データ(1): `['チケットクラス', '年齢', '同乗1', '同乗2', '運賃', 'female'] = [3, 27, 1, 0, 8, True]`
  * 予測する客船データ(2): `['チケットクラス', '年齢', '同乗1', '同乗2', '運賃', 'female'] = [1, 41, 0, 1, 30, False]`
* 学習したモデルの評価を行う
  * 訓練データとテストデータに対する精度をそれぞれ計算する
  * 特徴量重要度を計算する

**［実習内容］**
* 以下の「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する

**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイル「Survived.csv」をDataFrameとして読み込んで，変数`df`に代入
* 3行目: `get_dummies`関数を使って，性別列（`df['性別']`）をダミー変数化し，その結果を変数`dummy`に代入
* 4行目: `concat`関数を使って，2つのDataFrame`df`と`dummy`を結合し，変数`df`に代入
* 5行目: チケットクラスが1，かつ生存が0，かつ年齢が欠損している（`NaN`の）データの年齢列の値を43にする
  * 条件式は`(df['チケットクラス'] == 1) & (df['生存'] == 0) & (df['年齢'].isnull())`
* 6行目: チケットクラスが1，かつ生存が1，かつ年齢が欠損している（`NaN`の）データの年齢列の値を35にする
  * 条件式は`(df['チケットクラス'] == 1) & (df['生存'] == 1) & (df['年齢'].isnull())`
* 7行目: チケットクラスが2，かつ生存が0，かつ年齢が欠損している（`NaN`の）データの年齢列の値を33にする
  * 条件式は`(df['チケットクラス'] == 2) & (df['生存'] == 0) & (df['年齢'].isnull())`
* 8行目: チケットクラスが2，かつ生存が1，かつ年齢が欠損している（`NaN`の）データの年齢列の値を25にする
  * 条件式は`(df['チケットクラス'] == 2) & (df['生存'] == 1) & (df['年齢'].isnull())`
* 9行目: チケットクラスが3，かつ生存が0，かつ年齢が欠損している（`NaN`の）データの年齢列の値を26にする
  * 条件式は`(df['チケットクラス'] == 3) & (df['生存'] == 0) & (df['年齢'].isnull())`
* 10行目: チケットクラスが3，かつ生存が1，かつ年齢が欠損している（`NaN`の）データの年齢列の値を20にする
  * 条件式は`(df['チケットクラス'] == 3) & (df['生存'] == 1) & (df['年齢'].isnull())`
* 11行目: 説明変数の列名を要素とするリスト`['チケットクラス', '年齢', '同乗1', '同乗2', '運賃', 'female']`を変数`x_cols`に代入
* 12行目: 目的変数の列名`'生存'`を変数`t_col`に代入
* 13行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 14行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 15行目: `model_selection`モジュールの`train_test_split`関数のインポート
* 16行目: `train_test_split`関数を使って説明変数`x`と目的変数`t`を訓練データ（80％）とテストデータ（20％）にそれぞれ分割
  * `random_state=random_seed`で乱数の種を指定する
* 17行目: `sklearn` (scikit-learn) の`tree`モジュールの`DecisionTreeClassifier`クラスをインポート
* 18行目: 分類木モデルの学習を行うためのオブジェクトを`DecisionTreeClassifier`クラスから生成し，変数`model_tree`に代入
  * `class_weight='balanced'`で不均衡データの影響を取り除く
  * `max_depth=5`で最大深さを5とする
  * `random_state=random_seed`で乱数の種を指定する
* 19行目: `fit`メソッドで分類木モデルの学習を実行
* 20行目: 新しい2つの説明変数（`['チケットクラス', '年齢', '同乗1', '同乗2', '運賃', 'female']`）のリスト`[[3, 27, 1, 0, 8, True], [1, 41, 0, 1, 30, False]]`を変数`newdata`に代入
* 21行目: `predict`メソッドを用いて，新たな説明変数`newdata`に対する予測を行い，その結果（`predict`メソッドの戻り値）を`print`関数で表示
  * `f'予測結果： {model_tree.predict(X = newdata)}'` は f-string
* 22行目: `model_tree.score(X = x_train, y = t_train)`で，訓練データから精度を計算して変数`score_train`に代入
* 23行目: `model_tree.score(X = x_test, y = t_test)`で，テストデータに対する精度を計算して変数`score_test`に代入
* 24行目: `print`関数とf-stringを使って，訓練データに対する精度（`score_train`）とテストデータに対する精度（`score_test`）を小数点以下3桁まで表示
  * 「`:.3f`」で，小数点以下3桁まで表示
* 25行目: `display`関数で，`model_tree`オブジェクトの`feature_importances_`属性（特徴量重要度）を表示
  * `DataFrame`関数で，`model_tree.feature_importances_` をDataFrameに変換
  * そのDataFrameの行名（`index`）を説明変数`x`の列名（`x.columns`），列名（`columns`）を「特徴量重要度」とした

  **［実行結果］**

  <img src="./fig/exercise_Survived_result.jpg" width="400">


In [ ]:
# 一連の分析
df = pd.read_csv('Survived.csv')

















newdata = [[3, 27, 1, 0, 8, True], [1, 41, 0, 1, 30, False]]
print(f'予測結果： {model_tree.predict(X=newdata)}')


print(f'訓練データの精度={score_train:.3f} / テストデータの精度={score_test:.3f}')
display(pd.DataFrame(model_tree.feature_importances_, index = x.columns, columns = ['特徴量重要度']))

# 分類木によるデータ分析例 その2

## 用いるデータと分析の目的
* 例として，csvファイル「employee_train.csv」を用いる
* このデータは，ある会社の従業員データベース一覧をイメージしている
  * 出典: 須出典:池田他: [実務で役立つPython機械学習入門 課題解決のためのデータ分析の基礎](https://www.shoeisha.co.jp/book/detail/9784798184890), 翔泳社, 2023
  * オリジナルのデータ: https://github.com/ml-pg-book/python-business-ml-starter
* 欠損値や外れ値はない
* 以降，このデータを「従業員データ」と呼ぶ


|**列名**| **意味** |
|:--|:--|
|leaving| 0: 離職してない / 1: 離職した |
|gender| 性別（女性 or 男性） |
|age| 年齢 |
|job_role| 職種 |
|job_satisfaction| 仕事に対する満足度（1～5の5段階評価） |
|environment_satisfaction| 職場環境に対する満足度（1～5の5段階評価） |
|over_time| 残業が多かったか（0: 少なかった / 1: 多かった） |
|income| 年収 |

* 従業員データを用いて，離職する（離職しそうな）従業員を予測することを考える
* そのために，分類木モデルを作成する
* これにより，どんなタイプの従業員が離職しやすいかがわかる

## データの確認
**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，csvファイル「employee_train.csv」をDataFrameとして読み込んで，変数`df`に代入
* 3行目: `display`関数で`df`の内容を表示

In [ ]:
# データの読み込み
df = pd.read_csv('employee_train.csv')
display(df)

## 「leaving」列のデータの割合
* ここでは，列「leaving」のデータを目的変数として，分類木モデルの学習を行う
* まず「leving」列のカテゴリ（0 or 1）がどのような割合になっているのかを確認する
  
**［以下のコードの処理内容］**
* 2行目: csvファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: 列「leaving」の値の頻度を抽出し，その結果を`display`関数で表示
  * `df['leaving'].value_counts()`で，DataFrameである`df`の列「leaving」にどんな値がそれぞれ何個あるのかを調べる
  * `value_counts`メソッドの戻り値がSeriesなので，pandasの`DataFrame`関数でDataFrameに変換 ⇒ `pd.DataFrame(df['leaving'].value_counts())`
  * 変換したDataFrameを`display`関数で表示
*  実行結果から，従業員データが不均衡データであることが確認できる

In [ ]:
# 各カテゴリのデータ数の確認
df = pd.read_csv('employee_train.csv')
display(pd.DataFrame(df['leaving'].value_counts()))

## 実習：一連の分析
**［実習内容］**
* 以下の「分析手順」と「以下のコードの処理内容」に従って，コードを完成させる
* 空行に適切なコードを記述する
  
**［分析手順］**
* ホールドアウト法による分類木モデルの学習を行う
  * 目的変数: leaving 
  * 説明変数（5つ）: age, job_satisfaction, environment_satisfaction, over_time, income
  * 訓練データ 70％，テストデータ 30％として分割
  * 不均衡データの影響を取り除く
  * 木の最大深さは 4 とする
  * 乱数の種（`random_state`）は，すべて共通コードで定義している`random_seed`を用いる
* 学習したモデルを使って予測を行う
  * 予測する従業員データ(1): `['age', 'job_satisfaction', 'environment_satisfaction', 'over_time', 'income'] = [30, 3, 1, 0, 726]`
  * 予測する従業員データ(2): `['age', 'job_satisfaction', 'environment_satisfaction', 'over_time', 'income'] = [50, 3, 3, 0, 986]`
* 学習したモデルの評価を行う
  * 訓練データとテストデータに対する精度をそれぞれ計算する
  * 特徴量重要度を計算する

**［以下のコードの処理内容］**
* 2行目: pandasの`read_csv`関数を使って，ファイルをDataFrameとして読み込んで，変数`df`に代入
* 3行目: 説明変数の列名を要素とするリスト`['age', 'job_satisfaction', 'environment_satisfaction', 'over_time', 'income']`を変数`x_cols`に代入
* 4行目: 目的変数の列名`'leaving'`を変数`t_col`に代入
* 5行目: DataFrame `df` から,`df[x_cols]`で，説明変数の列だけ取り出し変数`x`に代入
* 6行目: DataFrame `df` から,`df[t_col]`で目的変数の列だけ取り出し変数`t`に代入
* 7行目: `model_selection`モジュールの`train_test_split`関数のインポート
* 8行目: `train_test_split`関数を使って説明変数`x`と目的変数`t`を訓練データ（70％）とテストデータ（30％）にそれぞれ分割
* 9行目: `sklearn` (scikit-learn) の`tree`モジュールの`DecisionTreeClassifier`クラスをインポート
* 10行目: 分類木モデルの学習を行うためのオブジェクトを`DecisionTreeClassifier`クラスから生成し，変数`model_tree`に代入
* 11行目: `fit`メソッドで分類木モデルの学習を実行
* 12行目: 新しい2つの説明変数（`['age', 'job_satisfaction', 'environment_satisfaction', 'over_time', 'income']`）のリスト`[[30, 3, 1, 0, 726], [50, 3, 3, 0, 986]]`を変数`newdata`に代入
* 13行目: `predict`メソッドを用いて，新たな説明変数`newdata`に対する予測を行い，その結果（`predict`メソッドの戻り値）を`print`関数で表示
* 14行目: `model_tree.score(X = x_train, y = t_train)`で，訓練データに対する精度を計算して変数`score_train`に代入
* 15行目: `model_tree.score(X = x_test, y = t_test)`で，テストデータに対する精度を計算して変数`score_test`に代入
* 16行目: `print`関数とf-stringを使って，訓練データに対する精度（`score_train`）とテストデータに対する精度（`score_test`）を小数点以下3桁まで表示
* 17行目: `display`関数で，`model_tree`オブジェクトの`feature_importances_`属性（特徴量重要度）を表示
  
  **［実行結果］**

  <img src="./fig/exercise_employee_result.jpg" width="400">

In [ ]:
# 一連の分析
df = pd.read_csv('employee_train.csv')




from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier



print(f'予測結果： {model_tree.predict(X=newdata)}')


print(f'訓練データの精度={score_train:.3f} / テストデータの精度={score_test:.3f}')
display(pd.DataFrame(model_tree.feature_importances_, index=x.columns, columns=['特徴量重要度']))

# 参考文献・資料
* 松尾豊(監修): 東京大学のデータサイエンティスト育成講座, マイナビ出版, 2019
* 池田雄太郎, 田尻俊宗, 新保雄大: 実務で役立つPython機械学習入門 課題解決のためのデータ分析の基礎, 翔泳社, 2023
